In [1]:
import numpy as np
import pandas as pd
import h5py
import os, sys
import torch
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
from typing import Dict, List
sys.path.append('../')
from config import JUNCTIONS, get_file_paths, get_file_paths_list, reverse_complement
from get_logits import *

In [2]:
tokenizer = AutoTokenizer.from_pretrained('kuleshov-group/PlantCaduceus_l32')

In [3]:
labels_dict = {}
for idx, junction in enumerate(JUNCTIONS):
    label = pd.read_csv(get_file_paths('labels', '../../../results/pseudo_gene/splice_junctions/')[junction], sep='\t')
    for model in ['pcv1', 'pcv2_1', 'pcv2_2', 'pcv2_3', 'evo2_fwd', 'evo2_rc', 'gpn']:
        scores = get_averaged_probs(model, idx, tokenizer, base_dir='../../../results/pseudo_gene/outputs/', chunk_size=None, JUNCTIONS=JUNCTIONS)
        label[f'{model}_scores'] = scores

    if junction in ['start_sites', 'acceptor']:
        label['evo2_scores'] = label['evo2_rc_scores']
    else:
        label['evo2_scores'] = label['evo2_fwd_scores']
        
    labels_dict[junction] = label

In [5]:
colors = ['#1f77b466', '#6baed6', '#1f77b4b3', '#1f77b4ff',  '#808080', '#999999', '#d3d3d3']

model_preds = [
    ('pcv1_scores', 'PlantCAD'),
    ('pcv2_1_scores', 'PlantCAD2-S'),
    ('pcv2_2_scores', 'PlantCAD2-M'),
    ('pcv2_3_scores', 'PlantCAD2-L'),
    ('evo2_scores', 'Evo2'),
    ('gpn_scores', 'GPN'),
]
colors_use = colors[:len(model_preds)]
model_labels = [lbl for _, lbl in model_preds]

def to_binary(label_str):
    # 1 = Core, 0 = non-Core; ignore "Near-Core Gene"
    return 1 if ('Core' in label_str and 'Near' not in label_str) else 0


for idx, (dataset_name, df) in enumerate(labels_dict.items()):

    # filter & binarize
    subsets = df[df['Label'] != 'Near-Core Gene'].copy()
    y_true = subsets['Label'].apply(to_binary).values

    # Count positives (1s) and negatives (0s)
    num_positives = sum(y_true)
    num_negatives = len(y_true) - num_positives
    
    print(f"{dataset_name} Number of positives: {num_positives}")
    print(f"{dataset_name} Number of negatives: {num_negatives}")

start_sites Number of positives: 28291
start_sites Number of negatives: 8118
stop_sites Number of positives: 28291
stop_sites Number of negatives: 8118
donor Number of positives: 123183
donor Number of negatives: 21367
acceptor Number of positives: 123183
acceptor Number of negatives: 21367
